## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

## Load the dataset + small sanity check

In [ ]:
from prepare_dataset import PairedCIFAR10

In [ ]:
train_dataset = PairedCIFAR10("train")
tval_dataset = PairedCIFAR10("val")
test_dataset = PairedCIFAR10("test")

In [ ]:
print(f"Train dataset: {len(train_dataset)}")
print(f"Validation dataset: {len(val_dataset)}")
print(f"Test dataset: {len(test_dataset)}")

In [ ]:
img, cond, lbl = train_dataset[0]

print(f"Image shape: {image.shape}")
print(f"Condition shape: {cond.shape}")
print(f"Label: {lbl}")
print(f"Condition vector: {cond}")

## Combine conditions for further analysis

In [ ]:
train_cond = torch.stack([
    train_dataset[i][1]
    for i in range(len(train_dataset))
])

val_cond = torch.stack([
    val_dataset[i][1]
    for i in range(len(val_dataset))
])

print(train_cond.shape)
print(val_cond.shape)

In [ ]:
train_cond_np = train_cond.numpy()
val_cond_np = val_cond.numpy()

## Per-feature analysis

In [ ]:
stats = pd.DataFrame({
    "mean": train_cond_np.mean(axis=0),
    "std": train_cond_np.std(axis=0),
    "min": train_cond_np.min(axis=0),
    "max": train_cond_np.max(axis=0),
})

In [ ]:
stats.index = [f"feature_{i}" for i in range(16)]
print(stats)

## Plot and compare the distributions

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(14, 10))

for i, ax in enumerate(axes.flat):
    ax.hist(train_cond_np[:, i], bins=50)
    ax.set_title(f"Feature {i}")
    ax.set_xlabel("Value")
    ax.set_ylabel("Count")

plt.tight_layout()
plt.show()

## Train/Val distribution comparison

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(14, 10))

for i, ax in enumerate(axes.flat):
    ax.hist(
        train_cond_np[:, i],
        bins=40,
        alpha=0.5,
        density=True,
        label="Train"
    )

    ax.hist(
        val_cond_np[:, i],
        bins=40,
        alpha=0.5,
        density=True,
        label="Validation"
    )

    ax.set_title(f"Feature {i}")

axes[0, 0].legend()

plt.tight_layout()
plt.show()

## Checking feature correlation

In [ ]:
corr = np.corrcoef(train_cond_np, rowvar=False)

plt.figure(figsize=(10, 8))
plt.imshow(corr, vmin=-1, vmax=1)
plt.colorbar(label="Correlation")


plt.xticks(range(16), range(16))
plt.yticks(range(16), range(16))

plt.xlabel("Features")
plt.ylabel("Features")
plt.title("Conditioning feature correlation")

plt.show()

## Condition analysis by class

In [ ]:
train_labels = np.array([
    int(train_dataset[i][2])
    for i in range(len(train_dataset))
])

In [ ]:
class_means = []

class_number = int(train_labels.max(axis=0))

for label in range(class_number):
    class_cond = train_cond_np[train_labels == label]
    class_means.append(class_cond.mean(axis=0))

class_means = np.array(class_means)

In [ ]:
class_means.shape

In [ ]:
plt.figure(figsize=(12, 6))

plt.imshow(class_means, aspect="auto")
plt.colorbar(label="Mean feature value")

plt.xlabel("Conditioning feature")
plt.ylabel("Class")

plt.xticks(range(16))
plt.yticks(range(16))
plt.title("Mean conditioning vector by class")

plt.show()

## Conclusion